In [1]:
using FFTW, LinearAlgebra, NBInclude, Printf

function  DeltaLabattoIIIC2(z) # 2-stage Radau
    return [1 1-2*z
         -1 1]; 
end    

function  DeltaLabattoIIIC3(z) # 3-stage Radau
    return [3 4  -1-6*z
        -1  0 1
        1  -4 3]; 
end    

function  DeltaLabattoIIIC4(z) # 4-stage Radau
    a = √5;
return [6 (5/2)*(1+a)  -(5/2)*(-1 + a)  1-12*z
    (1/2)*(-1-a) 0 a (1/2)*(1 - a)
    (1/2)*(-1 + a) -a 0  (1/2)*(1 + a)
    -1 (5/2)*(-1+a) -(5/2)*(1+a)   6]
end

# compute RK convolution weights

function convw_rk(N,T,RK,Ks::Function) 

dt = T/N;

if  (RK==2)   

s=2;
        
A = [1/2 -1/2 ; 1/2 1/2] ; b = [1/2 , 1/2]; c = [0 , 1];
genfunc = "LabattoIIIC2"  

elseif  (RK==3)
    s=3;
    A = [1/6  -1/3 1/6; 1/6 5/12 -1/12; 1/6 2/3 1/6]; b = [1/6, 2/3, 1/6];
        c = [0, 1/2 , 1]; 
    genfunc = "LabattoIIIC3"

  elseif  RK==4  # Labatto IIIC order 6, stage order 4    
      s=4;      
    A = [1/12 -√5/12 √5/12 -1/12 ; 1/12 1/4 (10-7*√5)/60 √5/60 ; 1/12 (10+7*√5)/60 1/4  -√5/60 ; 1/12 5/12 5/12 1/12];
    b = [1/12, 5/12, 5/12, 1/12];
    c = [0 , (5-√5)/10 , (5+√5)/10 , 1];
   genfunc = "LabattoIIIC4"       
    
end

Nmax = 3*N;                       # Nmax Power of 2 bigger than 2*N;
Nmax = 2^ceil(log(Nmax)/log(2));

λ = (1e-15)^(1/(2*Nmax));               		 
kv = 0:Nmax-1;                           
xi =  λ * exp.(1*im*2*pi*kv/Nmax);

omegalong=zeros(ComplexF64,(s,s,Int(Nmax)))

omega  = zeros(ComplexF64,(s,s,N+1))
omega0 = zeros(ComplexF64,s,s)


for ll=1:Int(Nmax)
xx = xi[ll];
DV  =  getfield(@__MODULE__, Symbol("Delta", genfunc))(xx)

D, V = eigen(DV)
temp = diagm(Ks.(D/dt));
invV = inv(V);
omegalong[:,:,ll] = V*temp*invV
end

for kk =1:s
    for jj = 1:s
omegaV = fft(omegalong[kk,jj,:])
omegaV = λ .^(-kv[1:N+1]) .*omegaV[1:N+1]
omega0[kk,jj] = omegaV[1]/Nmax;
omega[kk,jj,:] = omegaV/Nmax;
    end
end
    return real(omega), c # return to RK convolution matrix weights W[:, :, 1], W[:, :, 2] ...
end

convw_rk (generic function with 1 method)

In [2]:
RK = 2;   Ks(y) = y^(0.5); N = 20; T = 20; 

W, c = convw_rk(N,T,RK, Ks);
W

2×2×21 Array{Float64, 3}:
[:, :, 1] =
  1.09868  0.45509
 -0.45509  1.09868

[:, :, 2] =
 -0.160899   -0.843533
 -0.0666464  -0.160899

[:, :, 3] =
 -0.0520062  -0.101991
 -0.0313018  -0.0520062

;;; … 

[:, :, 19] =
 -0.00184853  -0.00200333
 -0.001712    -0.00184853

[:, :, 20] =
 -0.00170439  -0.00183954
 -0.00158446  -0.00170439

[:, :, 21] =
 -0.00157806  -0.00169689
 -0.00147203  -0.00157806

In [9]:
using SpecialFunctions, NLsolve, ForwardDiff

f(t) = t^3  + 6 * t + (3.2 / gamma(0.5)) * t^2.5; # exact = t^3 for
α=0.5;q₀ = 0.0;  p₀ = 0.0

0.0

In [10]:
function IntegratorBDF(q₀ ,  p₀ , N ,  T)

 h = T/N;  ts = (0:N)*h;
    
L(k) = (q,v) -> (1/2) * v^2 - (1/2) * q^2 + q * f(k*h);

Ld(k) = ((x , y),) -> (h/2) * L(k)(x , (y-x) / h) + (h/2) * L(k+1)(y , (y-x) / h)

DL(i,k) = u -> ForwardDiff.gradient( Ld(k) , u )[i]  

q = [];   
 
DEL0 = u -> p₀ + DL(1,1)([q₀ ,u]) - 0.5 * h^(1-α) * W[1,:,1]' * [q₀,u] ;

init_geuss =  [q₀ + h *  p₀];     
    
r = nlsolve(u -> DEL0(u[1]), init_geuss; autodiff = :forward,ftol=1e-14)
    
append!(q,q₀, r.zero); 


for k in 3:N+1  
        
Qnew = u -> vcat(q[1:end-1],u);
        
DEL = (q,u) -> DL(2,k-2)([q[end-1],q[end]]) + DL(1,k-1)([q[end] ,u])- 0.5*h^(1-α)* sum(W[1,:,k-j]'*Qnew[j:j+1] for j in 1:k) - 0.5*h^(1-α)* sum(W[2,:,k-j]'*q[j:j+1] for j in 1:k) ; 
        
init_geuss = [q[end] + h *  p₀];
        
r = nlsolve(u -> DEL(q,u[1]), init_geuss; autodiff = :forward,ftol=1e-14)
#all_converged(res) =  r.f_converged
#@show all_converged(r)
append!(q, r.zero); 
                    
end

    return q;
    
end
    

IntegratorBDF (generic function with 1 method)

In [11]:
q₀ = 0.0 ;  p₀ = 0.0 ; α = 0.5 ; N = 100 ; T = 1; ts = (0:N)*T/N;

exact(t)=t^3;
#exact(t)=t^2.5;
q  = IntegratorBDF(q₀ , p₀ , N , T)

plot(ts, exact.(ts))

scatter!(ts, q)


LoadError: MethodError: no method matching getindex(::var"#52#65"{Vector{Any}}, ::UnitRange{Int64})

In [49]:
IntegratorBDF(q₀ ,  p₀ , N ,  T)

LoadError: MethodError: no method matching getindex(::var"#312#325"{Vector{Any}}, ::UnitRange{Int64})